# Topics of the 1914 Press

What was *L'Humanité* writing about in 1914, and how did the news turn that summer? Without a single label, BERTopic groups similar passages and describes each group by its most characteristic words.

1. Find the topics
2. Tune them
3. Follow them over time
4. Name them with an LLM

The passages' *embeddings* are precomputed (with Qwen3-Embedding-0.6B, truncated to 512 dimensions). The GPU only serves step 4's LLM: *Runtime > Change runtime type > T4 GPU*.

## Data

Ollama installs and downloads step 4's LLM in the background.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/embeddings-512.npy
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
%pip install -q bertopic

import os
import subprocess

MODEL = "qwen3:4b-instruct-2507-q4_K_M"
env = {**os.environ, "OLLAMA_NUM_PARALLEL": "8", "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
pull = subprocess.Popen(f"sleep 5 && ollama pull {MODEL}", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

In [ ]:
import numpy as np
import pandas as pd

passages = pd.read_parquet("passages.parquet")
embeddings = np.load("embeddings-512.npy").astype(np.float32)

# L'Humanité only (Le Figaro has only 24 issues), and a sample of 20,000 passages so
# that the computation takes a few minutes.
humanite = passages[passages.newspaper == "L'Humanité"].sample(20_000, random_state=0)
docs = humanite.text.tolist()
dates = humanite.date.tolist()
vectors = embeddings[humanite.index]
print(len(docs), vectors.shape)

## 1. Find the Topics

Create a `BERTopic` model with:

- `umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0)` (reduction to 5 dimensions, reproducible)
- `vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10)`: to **describe** topics, stop words get in the way (the only use in this course where we remove them; the corpus is French, so are its stop words)
- `min_topic_size=50`
- `language="french"`: otherwise BERTopic strips every accented character from the keywords ("grève" becomes "grve")

Fit it with `fit_transform(docs, vectors)` (the precomputed vectors), then display `get_topic_info()`: how many topics? What share of the passages has no topic (topic $-1$)? Read the keywords of the 15 largest topics, and two representative passages (`get_representative_docs`) of three of them.

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.fr.stop_words import STOP_WORDS
from umap import UMAP

STOP_WORDS = sorted(STOP_WORDS | {"qu", "mm", "mme"})

In [ ]:
# Your code here

### Solution

In [ ]:
topic_model = BERTopic(
    language="french",
    umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0),
    vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
    min_topic_size=50,
)
topics, _ = topic_model.fit_transform(docs, vectors)
info = topic_model.get_topic_info()
print(len(info) - 1, "topics,", f"{np.mean(np.array(topics) == -1):.0%} without a topic")
info.head(16)

In [ ]:
for topic in info.Topic[1:4]:
    print(topic, topic_model.get_topic(topic)[:8])
    for doc in topic_model.get_representative_docs(topic)[:2]:
        print("   ", doc[:200])

## 2. Tune the Topics

- Rerun with `min_topic_size=20`, then `200`: how many topics, what share without a topic?
- With the first model, reassign the passages without a topic to their nearest topic (`topic_model.reduce_outliers(docs, topics, strategy="embeddings", embeddings=vectors)`), then update the descriptions (`topic_model.update_topics(docs, topics=new_topics, vectorizer_model=...)`).
- Find two topics about the same thing and merge them (`topic_model.merge_topics(docs, [a, b])`).

In [ ]:
# Your code here

### Solution

In [ ]:
for size in (20, 200):
    model = BERTopic(
        language="french",
        umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0),
        vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
        min_topic_size=size,
    )
    found, _ = model.fit_transform(docs, vectors)
    print(f"min_topic_size={size}: {len(set(found)) - 1} topics, {np.mean(np.array(found) == -1):.0%} without a topic")

In [ ]:
new_topics = topic_model.reduce_outliers(docs, topics, strategy="embeddings", embeddings=vectors)
topic_model.update_topics(docs, topics=new_topics, vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10))
print(f"{np.mean(np.array(new_topics) == -1):.0%} without a topic after reassignment")
topics = new_topics

# For instance two topics about strikes, or about the Caillaux trial: pick them by reading
# get_topic_info()'s keywords.

## 3. Follow the Topics over Time

`topic_model.topics_over_time(docs, dates, nr_bins=52)` counts each topic's passages per week; `topic_model.visualize_topics_over_time(...)` plots them. Show the topics about the war (*guerre*), Jaurès and the Caillaux trial: write a function `topic_with(word)` that returns the topic with the largest share of passages containing `word`. What happens between July and August 1914?

In [ ]:
# Your code here

### Solution

In [ ]:
over_time = topic_model.topics_over_time(docs, dates, nr_bins=52)


def topic_with(word: str) -> int:
    mentions = pd.Series(topics)[[word in doc.lower() for doc in docs]].value_counts()
    share = (mentions / pd.Series(topics).value_counts()).drop(-1, errors="ignore")
    return int(share.idxmax())


chosen = [topic_with(word) for word in ("guerre", "jaurès", "caillaux")]
print(chosen)
topic_model.visualize_topics_over_time(over_time, topics=chosen)

# In March, Mme Caillaux kills Gaston Calmette, the editor of Le Figaro; her trial fills
# July. Jaurès is assassinated on July 31. From August on, the war and the reports from the
# front dominate (look at the largest topic too, the military operations).

## 4. Name the Topics with an LLM

Keywords describe a topic, but don't name it. For each of the 20 largest topics, give the LLM its 10 keywords and its 3 representative passages (their first 300 characters), and ask it for a short name, 3 to 6 words, in English, in the `TopicName` format (spell the format out in the prompt: the model doesn't see the schema). Send the 20 requests in parallel (`ThreadPoolExecutor(8)`), set the names with `topic_model.set_topic_labels(names)` (a topic → name dictionary), and display `get_topic_info()`. Are the names right? Which one would you fix by hand?

In [ ]:
pull.wait()  # the model is downloaded

from concurrent.futures import ThreadPoolExecutor

from openai import OpenAI
from pydantic import BaseModel

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# A first call loads the model onto the GPU, where it stays: before any parallel work.
client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Hello"}], max_tokens=1)
print(MODEL, "ready")


class TopicName(BaseModel):
    name: str

In [ ]:
# Your code here

### Solution

In [ ]:
def name_topic(topic: int) -> str:
    keywords = ", ".join(word for word, _ in topic_model.get_topic(topic)[:10])
    examples = "\n\n".join(doc[:300] for doc in topic_model.get_representative_docs(topic)[:3])
    response = client.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": (
            "Here is a topic of the 1914 French press, found automatically.\n"
            f"Its keywords: {keywords}\n\nThree representative passages:\n{examples}\n\n"
            "Give it a short, precise name, 3 to 6 words, in English.\n"
            # The schema constrains the output, but the model doesn't see it: without this
            # reminder, it sometimes wanders off into whitespace before the JSON.
            'Answer with a JSON object: {"name": "..."}'
        )}],
        response_format=TopicName,
        temperature=0,
        max_tokens=100,
    )
    return response.choices[0].message.parsed.name


largest = [topic for topic in topic_model.get_topic_info().Topic if topic != -1][:20]
with ThreadPoolExecutor(8) as pool:
    names = dict(zip(largest, pool.map(name_topic, largest)))
topic_model.set_topic_labels(names)
topic_model.get_topic_info()[["Topic", "Count", "CustomName", "Representation"]].head(21)

# BERTopic can also do it itself: representation_model=bertopic.representation.OpenAI(client,
# model=MODEL), with any OpenAI-compatible client. Always read the names over: an LLM names a
# clear-cut topic well, a catch-all topic badly.